# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: MIT-0

# Automated Blood Sample Quality Control — Pipeline Validation Walkthrough

> **Illustrative use only.** This notebook treats platelet presence as a stand-in "defect" class purely to demonstrate the pipeline on a real public dataset. It is not a clinical or diagnostic application, and the labels do not reflect actual blood-sample quality.

This notebook serves two purposes:

1. **Inspection walkthrough** — run it start-to-finish against a deployed stack. Every cell executing without error exercises the path it demonstrates (pipeline execution, registry retrieval, ONNX inference) and shows what each step produced. It is not a test suite: see "What was **not** asserted" at the end for the two things a clean run does NOT establish. The notebook ships without saved outputs; execute it in your own account to produce them.
2. **Usage guide** — with code cells collapsed, this reads as a clean research report showing how to execute the pipeline, retrieve models, and run inference.

**Dataset:** [BCCD](https://github.com/Shenggan/BCCD_Dataset) (Blood Cell Count and Detection) — 364 blood smear microscopy images, MIT license.
**Classes:** RBC (red blood cells), WBC (white blood cells), Platelets.
**Models:**
- **Anomaly detector** (convolutional autoencoder) — trained on normal slides (RBC+WBC only), flags anomalous slides via reconstruction error
- **Classifier** (ResNet18) — classifies slides as good vs defect (Platelets present)

Both models are evaluated against quality gates -- anomaly AUROC, and classifier balanced accuracy AND defect recall -- and, once those pass, are registered in the SageMaker Model Registry with their ONNX artifacts. The README's Pipeline Parameters table lists every shipped default; `config.py` holds the anomaly AUROC floor specifically, and explains why it is a sample floor rather than a production bar.

---

## 0 — Prerequisites: Download the BCCD Dataset

The inference sections below use real BCCD test images from a local download.
This step clones the dataset, converts it to the anomalib layout, and uploads
tarballs to the pipeline S3 bucket so the pipeline execution in Section 1 can
train on real data. Run it once before executing the rest of the notebook.

An existing publication can be reused **for the same deployment** -- the prefixes are
content-derived, so re-publishing the same data changes nothing. If you switch to a
different account or Region, publish into a **fresh output directory** and point
Section 1's `index_path` at it: Section 1 refuses an index naming another deployment's
bucket rather than starting a run the execution role cannot read.

In [ ]:
import subprocess
import sys

# subprocess.run(check=True), NOT `!python`: a shell escape does not turn a nonzero
# exit into a Python error, so a failed publication read as success here and the
# cells below consumed a stale index. sys.executable is this kernel's interpreter.
subprocess.run(
    [
        sys.executable,
        "../scripts/download_bccd.py",
        "--output-dir", "../data/bccd",
        "--upload-s3",
    ],
    check=True,
)

In [ ]:
import json
import shutil
import tarfile
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
from IPython.display import Markdown, display
from PIL import Image

# Use the project's shared resolution rather than a direct boto3 call, so this
# notebook cannot target a different Region than app.py and the helper scripts.
# Resolving independently meant AWS_REGION=us-east-2 with a profile Region of
# us-west-2 gave the notebook a different answer than the deployment.
import sys

sys.path.insert(0, str(Path("..").resolve()))
from aws_env import resolve_region  # noqa: E402
from config import PUBLICATION_INDEX_NAME  # noqa: E402

REGION = resolve_region()
sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)


def get_stack_outputs(stack_name):
    resp = cfn.describe_stacks(StackName=stack_name)
    return {o["OutputKey"]: o["OutputValue"] for o in resp["Stacks"][0]["Outputs"]}


outputs = get_stack_outputs("SageMakerCVPipelineStack")
PIPELINE_NAME = outputs["CVPipelineName"]
BUCKET = outputs["CVBucketName"]

print(f"Pipeline:  {PIPELINE_NAME}")
print(f"S3 Bucket: {BUCKET}")
print(f"Region:    {REGION}")
print(f"Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")


## 1 — Pipeline Execution

Start a pipeline execution trained on **real BCCD blood cell images** uploaded to S3.
The `TrainingDataS3Uri` parameter tells the preprocessing step to download real data instead of generating synthetic images.

The pipeline runs 13 steps:
preprocess → fetch latest model (x2) → train (x2) → evaluate (x2) → quality gate (x2) → Neo compile (x2) → register (x2).

Measured runtime: ~15 minutes end-to-end on `ml.m5.xlarge` with synthetic data (Jul 2026, us-east-2), including both Neo compilations; real BCCD data adds download/upload time.

In [ ]:
# Datasets are published under IMMUTABLE, content-derived prefixes
# (cv-pipeline/datasets/<dataset_id>), so there is no "s3://<bucket>/data" to point
# at -- and the preprocessing step now refuses a prefix with no _READY marker
# rather than training on a possibly half-uploaded dataset. Read the exact URI
# from the publication index that Section 0 wrote.
index_path = Path("../data/bccd") / PUBLICATION_INDEX_NAME
if not index_path.is_file():
    raise RuntimeError(
        f"{index_path} not found. Run the download cell in Section 0 with "
        f"--upload-s3 first; it writes this index."
    )
publication_index = json.loads(index_path.read_text())
cameras = publication_index["cameras"]

# The single-dataset run publishes under the key "default".
DATASET_KEY = "default" if "default" in cameras else sorted(cameras)[0]
TRAINING_DATA_S3_URI = cameras[DATASET_KEY]["s3_uri"]
if not TRAINING_DATA_S3_URI:
    raise RuntimeError(
        f"Dataset {DATASET_KEY} was built locally but never uploaded. Re-run the "
        f"Section 0 cell with --upload-s3."
    )

# The index is a LOCAL file and this checkout can outlive a deployment. Its s3_uri may name
# another account's or Region's bucket, and the next cell would submit it as a billed run --
# which the bucket-scoped execution role then fails on, inside preprocessing. Compare parsed
# bucket identity, not a string prefix.
from urllib.parse import urlsplit

_published = urlsplit(TRAINING_DATA_S3_URI)
if _published.scheme != "s3" or _published.netloc != BUCKET:
    raise RuntimeError(
        f"Publication index {index_path} points at {TRAINING_DATA_S3_URI!r}, but this "
        f"deployment uses bucket {BUCKET!r}. Publish into THIS deployment and select its "
        f"index before starting the pipeline -- publish to a fresh output directory rather "
        f"than reusing one from another deployment."
    )

DATASET_ID = cameras[DATASET_KEY]["dataset_id"]
print(f"Dataset:   {DATASET_KEY} (dataset_id={DATASET_ID})")
print(f"S3 URI:    {TRAINING_DATA_S3_URI}")

resp = sm.start_pipeline_execution(
    PipelineName=PIPELINE_NAME,
    PipelineParameters=[
        {"Name": "TrainingInstanceType", "Value": "ml.m5.xlarge"},
        {"Name": "NumSamples", "Value": "400"},
        {"Name": "TrainingDataS3Uri", "Value": TRAINING_DATA_S3_URI},
        {"Name": "CameraId", "Value": DATASET_KEY},
    ],
)
EXECUTION_ARN = resp["PipelineExecutionArn"]
print(f"Started:   {EXECUTION_ARN}")
print("Polling every 60s...\n")

while True:
    desc = sm.describe_pipeline_execution(PipelineExecutionArn=EXECUTION_ARN)
    execution_status = desc["PipelineExecutionStatus"]
    elapsed = ""
    if "CreationTime" in desc:
        dt = (datetime.now(timezone.utc) - desc["CreationTime"].replace(tzinfo=timezone.utc)).total_seconds()
        elapsed = f" ({int(dt)}s elapsed)"
    print(f"  {time.strftime('%H:%M:%S')} - {execution_status}{elapsed}")
    if execution_status not in ("Executing", "Stopping"):
        break
    time.sleep(60)

print(f"\nPipeline finished: {execution_status}")
assert execution_status == "Succeeded", f"Pipeline failed with status: {execution_status}"


### Pipeline Step Results

Verify all 13 steps completed successfully.

In [ ]:
# Paginated: ListPipelineExecutionSteps returns NextToken, so a single call can
# truncate the step list -- and the model selection below reads the registration
# steps out of it, so a truncated list would silently lose a registered model.
steps = []
_steps_paginator = sm.get_paginator("list_pipeline_execution_steps")
for _page in _steps_paginator.paginate(PipelineExecutionArn=EXECUTION_ARN):
    steps.extend(_page["PipelineExecutionSteps"])
steps.sort(key=lambda s: s.get("StartTime", datetime.min.replace(tzinfo=timezone.utc)))

rows = []
for step in steps:
    name = step["StepName"]
    status = step["StepStatus"]
    start = step.get("StartTime")
    end = step.get("EndTime")
    duration = f"{(end - start).total_seconds():.0f}s" if start and end else "—"
    rows.append(f"| {name} | {status} | {duration} |")

table = "| Step | Status | Duration |\n|---|---|---|\n" + "\n".join(rows)
display(Markdown(table))

## 2 — Model Registry

The pipeline registers both models in the SageMaker Model Registry with quality metrics attached.
Each model version includes the ONNX artifact and the evaluation report that passed the quality gate.

In [ ]:
# The gate verdict is NOT recomputed here. The pipeline already applied it: a package
# reaching Approved/Completed passed the gate for the execution being inspected.
# Re-implementing the comparison meant this notebook carried its own copy of the
# threshold, which then disagreed with the shipped default and rendered 'fail' for a
# model that passes.
from config import ANOMALY_MODEL_GROUP, CLASSIFIER_MODEL_GROUP  # noqa: E402

MODEL_GROUPS = [ANOMALY_MODEL_GROUP, CLASSIFIER_MODEL_GROUP]


# In a fleet deployment every camera registers into the same group, so
# filter on the CameraId lineage metadata; taking the newest package
# unfiltered could hand you another camera's model.
# Matches the CameraId the run in Section 1 used.
CAMERA_ID = DATASET_KEY


def registered_by_this_execution():
    """Package ARNs the SELECTED execution registered, by model package group.

    Read from each step's Metadata.RegisterModel.Arn rather than searched for. The
    previous version independently picked the newest Approved package for the camera,
    which is not necessarily this execution's: if another execution registers for the
    same camera, this notebook loaded THAT model while reporting the run above, and
    the two branches could come from different executions. Nothing tied the status,
    the metrics and the inference results it presents together.
    """
    by_group = {}
    for step in steps:
        arn = step.get("Metadata", {}).get("RegisterModel", {}).get("Arn")
        if not arn:
            continue
        # arn:aws:sagemaker:<region>:<acct>:model-package/<group>/<version>
        group = arn.split("model-package/", 1)[1].rsplit("/", 1)[0]
        by_group.setdefault(group, []).append(arn)
    return by_group


REGISTERED = registered_by_this_execution()


def get_model_from_this_execution(group_name, camera_id=CAMERA_ID):
    """Describe the package THIS execution registered into *group_name*.

    The approval and completion checks are unchanged and still read the DESCRIPTION
    rather than a list page, because approval can change between the two: rejecting a
    model must stop this notebook from serving it.
    """
    arns = REGISTERED.get(group_name, [])
    if not arns:
        raise RuntimeError(
            f"Execution {EXECUTION_ARN} registered no model into {group_name}. "
            f"Registered groups: {sorted(REGISTERED) or 'none'}. This notebook "
            f"reports on that execution, so it will not fall back to a package from "
            f"a different one."
        )
    if len(arns) > 1:
        raise RuntimeError(
            f"Execution {EXECUTION_ARN} registered {len(arns)} packages into "
            f"{group_name}: {arns}. Which one the report describes would be "
            f"arbitrary."
        )
    desc = sm.describe_model_package(ModelPackageName=arns[0])
    actual_camera = desc.get("CustomerMetadataProperties", {}).get("CameraId")
    if actual_camera != camera_id:
        raise RuntimeError(
            f"{arns[0]} carries CameraId={actual_camera!r}, not {camera_id!r}"
        )
    if desc.get("ModelApprovalStatus") != "Approved":
        raise RuntimeError(
            f"{arns[0]} is {desc.get('ModelApprovalStatus')!r}, not Approved. A "
            f"rejected model must not be served."
        )
    if desc.get("ModelPackageStatus") != "Completed":
        raise RuntimeError(
            f"{arns[0]} has status {desc.get('ModelPackageStatus')!r}, not Completed"
        )
    return desc


# Absent metrics are reported as unavailable, NOT as an empty dict. Returning {} let
# the summary below print metric values pulled from nothing, so a model registered
# without a metrics artifact was reported as though its metrics had been read.
METRICS_UNAVAILABLE = {"__unavailable__": True}


def metric(metrics, name):
    """A metric value for the summary, or an explicit marker."""
    if metrics.get("__unavailable__"):
        return "unavailable"
    return metrics.get(name, "unavailable")


def fetch_metrics(desc):
    uri = desc.get("ModelMetrics", {}).get("ModelQuality", {}).get("Statistics", {}).get("S3Uri")
    if not uri:
        print(f"  {desc['ModelPackageArn']}: no ModelQuality statistics recorded")
        return METRICS_UNAVAILABLE
    bucket = uri.split("/")[2]
    key = "/".join(uri.split("/")[3:])
    return json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())


anomalib_desc = get_model_from_this_execution(ANOMALY_MODEL_GROUP)
classifier_desc = get_model_from_this_execution(CLASSIFIER_MODEL_GROUP)
anomalib_metrics = fetch_metrics(anomalib_desc)
classifier_metrics = fetch_metrics(classifier_desc)

rows = []
rows.append("| Model | Version | Approval | Key Metric | Value | Gate |")
rows.append("|---|---|---|---|---|---|")

av = anomalib_desc["ModelPackageArn"].split("/")[-1]
auroc = ("unavailable" if anomalib_metrics.get("__unavailable__")
         else anomalib_metrics.get("binary_classification_metrics", {})
         .get("auc", {}).get("value", "unavailable"))
auroc_pass = "passed in selected execution"
rows.append(f"| Anomaly detector | v{av} | {anomalib_desc['ModelApprovalStatus']} | AUROC | {auroc} | {auroc_pass} |")

cv = classifier_desc["ModelPackageArn"].split("/")[-1]
balanced_accuracy = metric(classifier_metrics, "balanced_accuracy")
defect_recall = metric(classifier_metrics, "defect_recall")
bal_pass = "passed in selected execution"
rows.append(f"| Classifier | v{cv} | {classifier_desc['ModelApprovalStatus']} | Balanced accuracy | {balanced_accuracy} | {bal_pass} |")
recall_pass = "passed in selected execution"
rows.append(f"| Classifier | v{cv} | {classifier_desc['ModelApprovalStatus']} | Defect recall | {defect_recall} | {recall_pass} |")

display(Markdown("\n".join(rows)))


## 3 — Download ONNX Models

Each registered model stores a `model.tar.gz` containing the exported ONNX artifact.
We download both models for local inference.

In [ ]:
TMP = Path(tempfile.mkdtemp(prefix="fleet-cv-report-"))


def download_onnx(desc, dest_dir):
    uri = desc["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]
    bucket = uri.split("/")[2]
    key = "/".join(uri.split("/")[3:])
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    tar_path = dest / "model.tar.gz"
    s3.download_file(bucket, key, str(tar_path))
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(dest, filter="data")
    onnx_path = next(dest.rglob("*.onnx"))
    size_kb = onnx_path.stat().st_size / 1024
    return onnx_path, size_kb


anomalib_onnx, anomalib_size = download_onnx(anomalib_desc, TMP / "anomalib")
classifier_onnx, classifier_size = download_onnx(classifier_desc, TMP / "classifier")

display(Markdown(
    f"| Model | File | Size |\n|---|---|---|\n"
    f"| Anomaly detector | `{anomalib_onnx.name}` | {anomalib_size:.0f} KB |\n"
    f"| Classifier | `{classifier_onnx.name}` | {classifier_size:.0f} KB |"
))

## 4 — Anomaly Detection Inference

The anomaly model is a convolutional autoencoder trained on "normal" blood smear slides
(containing only RBC and WBC). It learns to reconstruct normal slides — high reconstruction
error signals an anomalous sample (e.g., unexpected Platelets).

We test with real BCCD images: one normal slide and one containing Platelets.

In [ ]:
IMAGE_SIZE = 256
sess = ort.InferenceSession(str(anomalib_onnx), providers=["CPUExecutionProvider"])
inp_meta = sess.get_inputs()[0]
out_meta = sess.get_outputs()[0]


def anomaly_score(img):
    arr = np.array(img.resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.float32).transpose(2, 0, 1) / 255.0
    batch = arr[np.newaxis, ...]
    recon = sess.run([out_meta.name], {inp_meta.name: batch})[0]
    # Validate before scoring. A broadcastable shape like (1,1,1,1) gives a
    # plausible-looking MSE while comparing the image against something other than
    # its own reconstruction, and a non-finite output gives a NaN score that no
    # threshold can order. The pipeline's evaluator refuses both; this cell did not.
    if recon.shape != batch.shape:
        raise RuntimeError(
            f"Anomaly ONNX output shape {recon.shape} != input {batch.shape}"
        )
    if not np.isfinite(recon).all():
        raise RuntimeError("Anomaly ONNX output contains non-finite values")
    score = float(np.mean((recon - batch) ** 2))
    if not np.isfinite(score):
        raise RuntimeError(f"Reconstruction error {score} is not finite")
    return score


# Load real BCCD test images from the dataset Section 0 published. The path comes
# from the publication index: datasets live under an immutable
# <output>/<camera>/<dataset_id>/ directory, not the old flat ../data/bccd/anomalib.
# BCCD images are JPEGs and are now preserved as .jpg rather than being copied into
# .png filenames, so match every supported suffix instead of one.
IMAGE_SUFFIXES = (".png", ".jpg", ".jpeg")


def list_images(folder):
    return sorted(p for p in folder.iterdir()
                  if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)


bccd_test_anomalib = Path(cameras[DATASET_KEY]["local_path"]) / "test"
good_images = list_images(bccd_test_anomalib / "good")
defect_images = list_images(bccd_test_anomalib / "defect")
if not good_images or not defect_images:
    raise RuntimeError(
        f"No test images under {bccd_test_anomalib}. Re-run the Section 0 download."
    )

# Pick representative samples
normal_img = Image.open(good_images[0]).convert("RGB")
anomalous_img = Image.open(defect_images[0]).convert("RGB")

score_normal = anomaly_score(normal_img)
score_anomalous = anomaly_score(anomalous_img)
ratio = score_anomalous / score_normal if score_normal > 0 else float("inf")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(normal_img)
axes[0].set_title(f"Normal (RBC + WBC only)\nerror = {score_normal:.6f}", fontsize=10)
axes[0].axis("off")
axes[1].imshow(anomalous_img)
axes[1].set_title(f"Anomalous (Platelets present)\nerror = {score_anomalous:.6f}", fontsize=10)
axes[1].axis("off")
fig.suptitle(f"Anomaly Detection on BCCD Blood Smear Images — {ratio:.1f}x error difference", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Normal score:    {score_normal:.6f}  ({good_images[0].name})")
print(f"Anomalous score: {score_anomalous:.6f}  ({defect_images[0].name})")
print(f"Ratio:           {ratio:.1f}x")
print(f"\nTest set: {len(good_images)} normal, {len(defect_images)} anomalous slides")


## 5 — Classifier Inference

The ResNet18 classifier labels slides as **good** (RBC+WBC only) or **defect** (Platelets present).
We run ONNX inference on one image from each class and compare the predictions.

In [ ]:
CLASSIFIER_IMAGE_SIZE = 224
CLASS_NAMES = ["good", "defect"]

clf_sess = ort.InferenceSession(str(classifier_onnx), providers=["CPUExecutionProvider"])
clf_inp = clf_sess.get_inputs()[0]
clf_out = clf_sess.get_outputs()[0]


def classify(img):
    arr = np.array(
        img.resize((CLASSIFIER_IMAGE_SIZE, CLASSIFIER_IMAGE_SIZE)), dtype=np.float32
    ).transpose(2, 0, 1) / 255.0
    batch = arr[np.newaxis, ...]
    raw = clf_sess.run([clf_out.name], {clf_inp.name: batch})[0]
    # Validate the COMPLETE returned array before selecting from it. Indexing [0][0]
    # first discarded everything outside the first row, so a response such as
    # [[10, 0], [NaN, NaN]] became a confident "good" prediction: the invalid part
    # was thrown away before anything looked at it. argmax over NaN also silently
    # returns class 0, which is why the shape must be exact rather than merely
    # broadcastable.
    expected = (1, len(CLASS_NAMES))
    if raw.shape != expected or not np.isfinite(raw).all():
        raise RuntimeError(
            f"Classifier ONNX output is unusable: shape {raw.shape}, expected "
            f"{expected}; {int((~np.isfinite(raw)).sum())} non-finite value(s)."
        )
    logits = raw[0]
    probs = np.exp(logits - np.max(logits))
    probs /= probs.sum()
    pred = int(np.argmax(probs))
    return CLASS_NAMES[pred], float(probs[pred])



pred_good, conf_good = classify(normal_img)
pred_defect, conf_defect = classify(anomalous_img)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(normal_img)
axes[0].set_title(f"True: good\nPredicted: {pred_good} ({conf_good:.2f})", fontsize=10)
axes[0].axis("off")
axes[1].imshow(anomalous_img)
axes[1].set_title(f"True: defect\nPredicted: {pred_defect} ({conf_defect:.2f})", fontsize=10)
axes[1].axis("off")
fig.suptitle("ResNet18 Defect Classification on BCCD Blood Smear Images", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Good slide   -> predicted {pred_good} (confidence {conf_good:.4f})")
print(f"Defect slide -> predicted {pred_defect} (confidence {conf_defect:.4f})")


## 6 - Summary

This notebook is an **inspection walkthrough**, not a validation suite. Reaching this
cell means each step above ran without raising -- it does not mean the pipeline was
verified.

What was actually asserted:

- the pipeline execution reached `Succeeded` (Section 1);
- an **Approved**, `Completed` model package exists for this camera in each model
  group (**Section 2**). `Gate` in that table reports the result the *selected
  execution* established: registration only happens after the pipeline's own quality
  gate passes, so an Approved package means that gate passed. This notebook does not
  re-evaluate the threshold. Where the evaluation report could not be read, the
  **Value** column shows `unavailable` -- that is a missing metric, not a failed gate;
- both ONNX artifacts loaded and produced finite output (Sections 4 and 5).

What was **not** asserted:

- that the two images shown are classified correctly. They are illustrative single
  samples; aggregate quality is what the pipeline's quality gates measure, on the
  full test split.
- that the compiled SageMaker Neo artifact runs on any device. This notebook runs
  the **ONNX** model through ONNX Runtime, not the compiled artifact through DLR.
  See the README section "What is and is not tested".


In [ ]:
summary = f"""| Check | Result |
|---|---|
| Pipeline execution | {execution_status} |
| All steps completed | {len(steps)} steps |
| Anomaly detector registered | v{av}, AUROC={auroc} |
| Classifier registered | v{cv}, balanced_accuracy={balanced_accuracy}, defect_recall={defect_recall} |
| Anomaly ONNX inference | {ratio:.1f}x anomaly separation |
| Classifier ONNX inference | good -> {pred_good}, defect -> {pred_defect} |
| Report generated | {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')} |"""

display(Markdown(summary))

# Cleanup
shutil.rmtree(TMP, ignore_errors=True)
print(f"\nTemporary files cleaned up. Report complete.")